# Agent-Based Wealth Distribution Simulation

This notebook provides an interactive environment to explore the wealth distribution dynamics model.

## Model Overview

The simulation models a population of agents whose wealth evolves over time according to:
1. **Stochastic shocks**: Individual wealth changes drawn from a probability distribution
2. **Aggregate growth constraint**: Total wealth grows by a fixed amount each time step
3. **Non-negativity**: Agents cannot have negative wealth (bankruptcy floor)

This creates an interesting tension between systematic growth (lifting all boats) and random variation (creating winners and losers).

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.simulation import Simulation
from src.visualization import (
    plot_wealth_distribution,
    plot_lorenz_curve,
    plot_wealth_evolution,
    plot_gini_evolution,
    create_dashboard,
    plot_distribution_snapshots,
)

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 1. Basic Simulation

Let's start with a simple simulation using default parameters.

In [ ]:
# Create and run simulation
sim = Simulation(
    num_agents=100,
    initial_wealth_min=0.0,
    initial_wealth_max=100.0,
    aggregate_growth_per_step=10.0,
    wealth_change_std=5.0,
    random_seed=42,
)

print("Initial state:")
print(sim)
print("\nInitial statistics:")
for key, value in sim.get_statistics().items():
    print(f"  {key:15s}: {value:10.2f}")

In [ ]:
# Run simulation
history = sim.run(num_steps=100, record_interval=1)

print("Final state:")
print(sim)
print("\nFinal statistics:")
for key, value in sim.get_statistics().items():
    print(f"  {key:15s}: {value:10.2f}")

## 2. Visualize Results

### 2.1 Comprehensive Dashboard

In [ ]:
# Create comprehensive dashboard
final_wealths = sim.get_wealth_array()
fig = create_dashboard(history, final_wealths)
plt.show()

### 2.2 Distribution Snapshots Over Time

In [ ]:
# Show distribution evolution at key time points
time_steps = [0, 25, 50, 75, 100]
fig = plot_distribution_snapshots(history, time_steps)
plt.show()

## 3. Parameter Sensitivity Analysis

Let's explore how different parameters affect wealth inequality.

### 3.1 Effect of Volatility (wealth_change_std)

In [ ]:
# Test different volatility levels
volatilities = [1.0, 5.0, 10.0, 20.0]
results = []

for vol in volatilities:
    sim = Simulation(
        num_agents=100,
        initial_wealth_min=0.0,
        initial_wealth_max=100.0,
        aggregate_growth_per_step=10.0,
        wealth_change_std=vol,
        random_seed=42,
    )
    history = sim.run(num_steps=100, record_interval=10)
    
    # Calculate final Gini
    final_gini = sim.get_statistics()['gini']
    results.append({'volatility': vol, 'final_gini': final_gini, 'history': history})
    
# Plot Gini evolution for different volatilities
fig, ax = plt.subplots(figsize=(12, 6))
for result in results:
    vol = result['volatility']
    history = result['history']
    
    # Calculate Gini for each step
    def calc_gini(wealths):
        wealths_sorted = np.sort(wealths)
        n = len(wealths)
        cumsum = np.cumsum(wealths_sorted)
        if cumsum[-1] == 0:
            return 0
        return (2 * np.sum((np.arange(n) + 1) * wealths_sorted)) / (n * cumsum[-1]) - (n + 1) / n
    
    gini_series = history.groupby('step')['wealth'].apply(calc_gini)
    ax.plot(gini_series.index, gini_series.values, linewidth=2, label=f'σ = {vol}')

ax.set_xlabel('Time Step')
ax.set_ylabel('Gini Coefficient')
ax.set_title('Effect of Volatility on Inequality')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("\nFinal Gini coefficients:")
for result in results:
    print(f"  σ = {result['volatility']:5.1f}: Gini = {result['final_gini']:.3f}")

### 3.2 Effect of Aggregate Growth Rate

In [ ]:
# Test different growth rates
growth_rates = [0.0, 5.0, 10.0, 20.0]
results = []

for growth in growth_rates:
    sim = Simulation(
        num_agents=100,
        initial_wealth_min=0.0,
        initial_wealth_max=100.0,
        aggregate_growth_per_step=growth,
        wealth_change_std=5.0,
        random_seed=42,
    )
    history = sim.run(num_steps=100, record_interval=10)
    
    stats = sim.get_statistics()
    results.append({
        'growth': growth,
        'final_total_wealth': stats['total_wealth'],
        'final_mean_wealth': stats['mean'],
        'final_gini': stats['gini'],
    })

# Display results
results_df = pd.DataFrame(results)
print("\nEffect of Growth Rate:")
print(results_df.to_string(index=False))

### 3.3 Effect of Population Size

In [ ]:
# Test different population sizes
population_sizes = [50, 100, 200, 500]
results = []

for pop in population_sizes:
    sim = Simulation(
        num_agents=pop,
        initial_wealth_min=0.0,
        initial_wealth_max=100.0,
        aggregate_growth_per_step=10.0,
        wealth_change_std=5.0,
        random_seed=42,
    )
    history = sim.run(num_steps=100, record_interval=10)
    
    final_gini = sim.get_statistics()['gini']
    results.append({'population': pop, 'final_gini': final_gini})

# Display results
results_df = pd.DataFrame(results)
print("\nEffect of Population Size:")
print(results_df.to_string(index=False))

## 4. Custom Experiments

Use the cells below to run your own custom experiments.

In [ ]:
# Create your own simulation
custom_sim = Simulation(
    num_agents=200,  # Modify parameters as desired
    initial_wealth_min=0.0,
    initial_wealth_max=100.0,
    aggregate_growth_per_step=15.0,
    wealth_change_std=8.0,
    random_seed=123,
)

custom_history = custom_sim.run(num_steps=200, record_interval=2)

# Visualize
custom_wealths = custom_sim.get_wealth_array()
fig = create_dashboard(custom_history, custom_wealths)
plt.show()

## 5. Data Export

Export simulation results for further analysis.

In [ ]:
# Save history to CSV
output_path = Path.cwd().parent / 'data' / 'simulation_results.csv'
output_path.parent.mkdir(exist_ok=True)
history.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

# Show sample of data
print("\nSample of exported data:")
print(history.head(10))